# SSS Marine Debris Detection — Train & Compare

## Models
| Model | Params | Architecture |
|-------|--------|-------------|
| YOLOv8n | 3.01M | Standard (baseline) |
| SS-YOLO | 1.66M | GhostConv + FastC2f (47% lighter) |
| YOLOv8-ESI | 3.18M | C2f + SE attention (better texture) |

## Dataset: E5 (E4 + Realistic SSS Noise)
- **Train**: 953 images (153 debris + 200 clean BG + 600 noisy BG)
- **Noise types**: speckle, nadir lines, acoustic shadows, brightness variation, seabed textures
- **Val/Test**: Clean (no augmentation) for fair evaluation

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 1: Setup
# ═══════════════════════════════════════════════════════════
!pip install ultralytics pandas matplotlib -q
!git clone https://github.com/YOUR_REPO/sonar-vision.git  # UPDATE THIS
%cd sonar-vision

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 2: Upload E5 dataset
# ═══════════════════════════════════════════════════════════
from google.colab import files
uploaded = files.upload()  # Upload e5.zip
!unzip -q e5.zip -d /content/

import os
print('\nE5 Dataset stats:')
for split in ['train', 'val', 'test']:
    imgs = [f for f in os.listdir(f'/content/e5/images/{split}') if f.endswith('.png')]
    lbls = [f for f in os.listdir(f'/content/e5/labels/{split}') if f.endswith('.txt')]
    pos = sum(1 for l in lbls if os.path.getsize(f'/content/e5/labels/{split}/{l}') > 0)
    noisy = sum(1 for i in imgs if '_N' in i)
    print(f'  {split}: {len(imgs)} images ({pos} debris, {len(imgs)-pos} bg, {noisy} noisy)')

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 3: Visualize the noise augmentation
# ═══════════════════════════════════════════════════════════
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

train_dir = '/content/e5/images/train'
bg_imgs = sorted([f for f in os.listdir(train_dir) if 'BG' in f and '_N' not in f])[:4]
noisy_imgs = sorted([f for f in os.listdir(train_dir) if '_N' in f])[:4]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, name in enumerate(bg_imgs):
    axes[0, i].imshow(np.array(Image.open(os.path.join(train_dir, name))), cmap='gray')
    axes[0, i].set_title(f'Clean BG: {name[:20]}', fontsize=9)
    axes[0, i].axis('off')
for i, name in enumerate(noisy_imgs):
    axes[1, i].imshow(np.array(Image.open(os.path.join(train_dir, name))), cmap='gray')
    axes[1, i].set_title(f'Noisy BG: {name[:20]}', fontsize=9)
    axes[1, i].axis('off')
plt.suptitle('E5 Dataset: Clean BG vs Noisy BG', fontsize=13)
plt.tight_layout()
plt.show()

## STEP 4: Train All Three Models

Uses E5 dataset (with noisy backgrounds). Each model gets 100 epochs.

In [ ]:
# ═══════════════════════════════════════════════════════════
# Load custom modules
# ═══════════════════════════════════════════════════════════
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.nn.modules.conv import Conv
from ultralytics.nn.modules.block import C2f, SPPF
from models.sss_custom_modules import (
    WaveletConv, FastC2f, GhostConv,
    DepthwiseSeparableConv, SEBlock, CBAM
)
from models.build_sss_models import build_ss_yolo, build_yolov8_esi_full, C2fWithSE
from ultralytics.models.yolo.detect.train import DetectionTrainer
print('✓ Custom modules loaded')

In [ ]:
# ═══════════════════════════════════════════════════════════
# Train YOLOv8n on E5
# ═══════════════════════════════════════════════════════════
print('\n' + '='*60)
print('Training: YOLOv8n (Baseline) on E5')
print('='*60)

model_v8n = YOLO('yolov8n.pt')
model_v8n.train(
    data='/content/e5/data.yaml',
    epochs=100, imgsz=512, batch=16, patience=30,
    lr0=0.005, lrf=0.01, warmup_epochs=3,
    mosaic=0.0, mixup=0.0,
    fliplr=0.0, flipud=0.0, degrees=0.0,
    translate=0.05, scale=0.2,
    name='yolov8n_e5', project='/content/runs', exist_ok=True, plots=True,
)

# Evaluate
val_v8n = model_v8n.val(data='/content/e5/data.yaml', split='val', conf=0.25)
test_v8n = model_v8n.val(data='/content/e5/data.yaml', split='test', conf=0.25)

print(f'\n✓ YOLOv8n on E5:')
print(f'  Val  — mAP50: {val_v8n.box.map50:.4f}, P: {val_v8n.box.mp:.4f}, R: {val_v8n.box.mr:.4f}')
print(f'  Test — mAP50: {test_v8n.box.map50:.4f}, P: {test_v8n.box.mp:.4f}, R: {test_v8n.box.mr:.4f}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# Train SS-YOLO on E5
# ═══════════════════════════════════════════════════════════
print('\n' + '='*60)
print('Training: SS-YOLO (GhostConv + FastC2f) on E5')
print('='*60)

ss_model = build_ss_yolo()

_orig = DetectionTrainer.get_model
def _patched_ss(self, cfg=None, weights=None, verbose=True):
    from ultralytics.nn.tasks import DetectionModel
    from ultralytics.utils import RANK
    model = self.set_model_names_for_load(
        DetectionModel(cfg, nc=self.data['nc'], ch=self.data['channels'], verbose=verbose and RANK == -1)
    )
    model.model = ss_model.model
    model.nc = 1
    model.names = {0: 'marine_debris'}
    return model
DetectionTrainer.get_model = _patched_ss

try:
    yolo_ss = YOLO('yolov8n.pt')
    yolo_ss.train(
        data='/content/e5/data.yaml',
        epochs=100, imgsz=512, batch=16, patience=30,
        lr0=0.005, lrf=0.01, warmup_epochs=3,
        mosaic=0.0, mixup=0.0,
        fliplr=0.0, flipud=0.0, degrees=0.0,
        translate=0.05, scale=0.2,
        name='ss_yolo_e5', project='/content/runs', exist_ok=True, plots=True,
    )
    val_ss = yolo_ss.val(data='/content/e5/data.yaml', split='val', conf=0.25)
    test_ss = yolo_ss.val(data='/content/e5/data.yaml', split='test', conf=0.25)
    print(f'\n✓ SS-YOLO on E5:')
    print(f'  Val  — mAP50: {val_ss.box.map50:.4f}, P: {val_ss.box.mp:.4f}, R: {val_ss.box.mr:.4f}')
    print(f'  Test — mAP50: {test_ss.box.map50:.4f}, P: {test_ss.box.mp:.4f}, R: {test_ss.box.mr:.4f}')
except Exception as e:
    print(f'✗ SS-YOLO failed: {e}')
    import traceback; traceback.print_exc()
finally:
    DetectionTrainer.get_model = _orig

In [ ]:
# ═══════════════════════════════════════════════════════════
# Train YOLOv8-ESI on E5
# ═══════════════════════════════════════════════════════════
print('\n' + '='*60)
print('Training: YOLOv8-ESI (SE Attention) on E5')
print('='*60)

esi_model = build_yolov8_esi_full()

_orig2 = DetectionTrainer.get_model
def _patched_esi(self, cfg=None, weights=None, verbose=True):
    from ultralytics.nn.tasks import DetectionModel
    from ultralytics.utils import RANK
    model = self.set_model_names_for_load(
        DetectionModel(cfg, nc=self.data['nc'], ch=self.data['channels'], verbose=verbose and RANK == -1)
    )
    model.model = esi_model.model
    model.nc = 1
    model.names = {0: 'marine_debris'}
    return model
DetectionTrainer.get_model = _patched_esi

try:
    yolo_esi = YOLO('yolov8n.pt')
    yolo_esi.train(
        data='/content/e5/data.yaml',
        epochs=100, imgsz=512, batch=16, patience=30,
        lr0=0.005, lrf=0.01, warmup_epochs=3,
        mosaic=0.0, mixup=0.0,
        fliplr=0.0, flipud=0.0, degrees=0.0,
        translate=0.05, scale=0.2,
        name='yolov8_esi_e5', project='/content/runs', exist_ok=True, plots=True,
    )
    val_esi = yolo_esi.val(data='/content/e5/data.yaml', split='val', conf=0.25)
    test_esi = yolo_esi.val(data='/content/e5/data.yaml', split='test', conf=0.25)
    print(f'\n✓ YOLOv8-ESI on E5:')
    print(f'  Val  — mAP50: {val_esi.box.map50:.4f}, P: {val_esi.box.mp:.4f}, R: {val_esi.box.mr:.4f}')
    print(f'  Test — mAP50: {test_esi.box.map50:.4f}, P: {test_esi.box.mp:.4f}, R: {test_esi.box.mr:.4f}')
except Exception as e:
    print(f'✗ YOLOv8-ESI failed: {e}')
    import traceback; traceback.print_exc()
finally:
    DetectionTrainer.get_model = _orig2

## STEP 5: Compare Results

In [ ]:
import pandas as pd

results = []
model_map = {'YOLOv8n': model_v8n, 'SS-YOLO': yolo_ss, 'YOLOv8-ESI': yolo_esi}
val_map = {'YOLOv8n': val_v8n, 'SS-YOLO': val_ss, 'YOLOv8-ESI': val_esi}
test_map = {'YOLOv8n': test_v8n, 'SS-YOLO': test_ss, 'YOLOv8-ESI': test_esi}

for name in ['YOLOv8n', 'SS-YOLO', 'YOLOv8-ESI']:
    val_r, test_r = val_map[name], test_map[name]
    p, r = test_r.box.mp, test_r.box.mr
    f1 = 2*p*r / max(p+r, 1e-8)
    n = sum(p_.numel() for p_ in model_map[name].model.parameters())
    results.append({
        'Model': name, 'Params': f'{n/1e6:.2f}M',
        'Val mAP50': f'{val_r.box.map50:.4f}',
        'Test mAP50': f'{test_r.box.map50:.4f}',
        'Test P': f'{p:.4f}', 'Test R': f'{r:.4f}',
        'Test F1': f'{f1:.4f}',
    })

df = pd.DataFrame(results)
print('\n' + '='*70)
print('RESULTS ON E5 (with noisy backgrounds)')
print('='*70)
print(df.to_string(index=False))
df.to_csv('/content/results_e5.csv', index=False)

## STEP 6: Stress Test on Heavily Noised Data

Apply extreme noise to the test set — simulating worst-case deployment.

In [ ]:
# ═══════════════════════════════════════════════════════════
# Stress test
# ═══════════════════════════════════════════════════════════
import shutil
from pathlib import Path
from scripts.generate_e5_noisy import apply_random_sss_noise

noisy_test = Path('/content/e5_stress_test')
(noisy_test / 'images').mkdir(parents=True, exist_ok=True)
(noisy_test / 'labels').mkdir(parents=True, exist_ok=True)

random.seed(99)
test_imgs = sorted([f for f in os.listdir('/content/e4/images/test') if f.endswith('.png')])

for name in test_imgs:
    orig = np.array(Image.open(f'/content/e4/images/test/{name}'))
    # Apply 5 passes of heavy noise
    noisy = orig.copy()
    for _ in range(5):
        noisy = apply_random_sss_noise(noisy, 'heavy')
    Image.fromarray(noisy).save(noisy_test / 'images' / name)
    lbl_src = f'/content/e4/labels/test/{name.replace(".png", ".txt")}'
    if os.path.exists(lbl_src):
        shutil.copy(lbl_src, noisy_test / 'labels' / name.replace('.png', '.txt'))
    else:
        (noisy_test / 'labels' / name.replace('.png', '.txt')).touch()

noisy_yaml = f"""path: {noisy_test}
train: images
val: images
test: images
names:
  0: marine_debris
"""
with open(noisy_test / 'data.yaml', 'w') as f:
    f.write(noisy_yaml)

print(f'Stress test: {len(test_imgs)} heavily-noised images')

# Evaluate all models
print('\n' + '='*60)
print('STRESS TEST: Heavy noise on test set')
print('='*60)

stress_results = []
for name, model in [('YOLOv8n', model_v8n), ('SS-YOLO', yolo_ss), ('YOLOv8-ESI', yolo_esi)]:
    r = model.val(data=str(noisy_test / 'data.yaml'), split='test', conf=0.25, verbose=False)
    p, rv = r.box.mp, r.box.mr
    f1 = 2*p*rv / max(p+rv, 1e-8)
    stress_results.append({'Model': name, 'mAP50': r.box.map50, 'P': p, 'R': rv, 'F1': f1})
    print(f'  {name}: mAP50={r.box.map50:.4f}, P={p:.4f}, R={rv:.4f}, F1={f1:.4f}')

pd.DataFrame(stress_results).to_csv('/content/stress_test_results.csv', index=False)
print('\n✓ Stress test complete')

In [ ]:
# ═══════════════════════════════════════════════════════════
# Per-target breakdown on noisy data
# ═══════════════════════════════════════════════════════════
print('\n' + '='*70)
print('PER-TARGET DETECTION ON NOISY DATA')
print('='*70)

for name, model in [('YOLOv8n', model_v8n), ('SS-YOLO', yolo_ss), ('YOLOv8-ESI', yolo_esi)]:
    print(f'\n--- {name} ---')
    preds = model.predict(source=str(noisy_test / 'images'), imgsz=512,
                          conf=0.25, save=False, verbose=False)
    
    target_stats = {}
    for r in preds:
        img_name = os.path.basename(str(r.path))
        parts = img_name.replace('.png','').split('_')
        tid = 'BG'
        for p in parts:
            if p.startswith('TGT'):
                tid = p
                break
        
        if tid not in target_stats:
            target_stats[tid] = {'detected': 0, 'total': 0, 'confs': []}
        target_stats[tid]['total'] += 1
        if len(r.boxes) > 0:
            target_stats[tid]['detected'] += 1
            target_stats[tid]['confs'].extend([float(c) for c in r.boxes.conf])
    
    print(f"  {'Target':>8} {'Imgs':>5} {'Det':>4} {'Rate':>7} {'AvgConf':>8}")
    for tid in sorted(target_stats.keys()):
        s = target_stats[tid]
        rate = s['detected'] / max(s['total'], 1)
        avg_c = np.mean(s['confs']) if s['confs'] else 0
        bar = '█' * int(rate * 15) + '░' * (15 - int(rate * 15))
        print(f'  {tid:>8} {s["total"]:>5} {s["detected"]:>4} {bar} {rate:>6.0%} {avg_c:>7.3f}')

## STEP 7: Export Best Model

In [ ]:
# ═══════════════════════════════════════════════════════════
# Export best model
# ═══════════════════════════════════════════════════════════
stress_df = pd.read_csv('/content/stress_test_results.csv')
best_name = stress_df.loc[stress_df['F1'].idxmax(), 'Model']
best_model = model_map[best_name]

print(f'Best model: {best_name}')
best_model.export(format='onnx', imgsz=512)

from google.colab import files
model_dir = f'runs/{best_name.lower().replace("-","")}_e5'
files.download(f'{model_dir}/weights/best.pt')
files.download(f'{model_dir}/weights/best.onnx')